# TruLens: from beginner to expert in LLM performance evals

A hands-on course in **TruLens 2.x** (OpenTelemetry-based) for engineers who build RAG systems and agents. Every concept comes with code you run, using a small datasheet Q&A app and a small tool-using agent, so the whole notebook costs only a few cents of API calls.

| Part | You will learn |
|---|---|
| **0. Mental model** | What TruLens is, its building blocks, and how an evaluation flows |
| **1. Beginner** | Sessions, instrumenting an app, recording, providers, the RAG triad, reading results, the dashboard |
| **2. Intermediate** | Instrumentation in depth, selectors, the metric catalog, customizing LLM judges, deterministic metrics, ground truth, groundedness, versions and experiments |
| **3. Advanced** | Agent (trace-level) evals, multi-turn conversations, evaluation modes, offline/batch evals, runs and regression testing, online sampling, guardrails, dataset curation, prompt registry, OpenTelemetry interop, cost |
| **4. Expert** | Meta-evaluation: validating and calibrating your judges, juries, few-shot optimization, bias checks, designing an eval suite, CI gating, production checklist |
| **5. Exercises** | Practice tasks to cement it |

> Version note: written against **TruLens 2.16** in OpenTelemetry mode (the default). Older tutorials use `Feedback`, `Select`, `TruCustomApp` and `Tru()`. Those are legacy names. This notebook uses the current `Metric`, `Selector`, `TruApp` and `TruSession`. `Feedback` still exists as a subclass of `Metric`.

**Prerequisites:** `uv sync`, the `.venv` kernel, and `OPENAI_API_KEY` in `.env`. No Qdrant or Phoenix is needed: this notebook is self-contained.

## 0. Mental model

```
           your app (RAG / agent / any Python)
                │  @instrument  → OpenTelemetry spans (RECORD_ROOT, RETRIEVAL, GENERATION, TOOL, AGENT, …)
                ▼
     ┌──────────────────────┐        one app invocation = one RECORD (a trace)
     │  TruApp (recorder)   │──────▶ records + spans stored by the TruSession (SQLite / Postgres / Snowflake)
     └──────────────────────┘
                │  Metrics = implementation(provider judge or Python fn) + Selectors (which span data feeds which argument)
                ▼
     ┌──────────────────────┐
     │  Metric evaluation   │──────▶ EVAL spans: score ∈ [0,1], explanation, cost  (per record or per conversation)
     └──────────────────────┘
                ▼
   leaderboard · records table · dashboard · run comparison / regression tests
```

| Concept | Class | What it is |
|---|---|---|
| Session | `TruSession` | Connection to the database and the evaluator. One per process. |
| App recorder | `TruApp` | Wraps your app object. Each call to its *main method* inside `with tru_app:` becomes a **record**. |
| Instrumentation | `@instrument(span_type=…, attributes=…)` | Turns a method call into an OTel span with typed attributes |
| Provider | `LLMProvider` subclasses | The **judge** model plus a library of evaluation prompts |
| Metric (feedback function) | `Metric` (alias `Feedback`) | `implementation` + `selectors` + optional aggregation → a score |
| Selector | `Selector` | Points a metric argument at data inside the trace (input, output, contexts, any span attribute, the whole trace, a conversation, a dataset column) |
| App version | `app_version=` | The unit you compare: prompt, model, retriever or agent variants |

# Part 1: Beginner

## 1.1 Setup and `TruSession`

`TruSession` is a singleton holding the database connection. SQLite works for local development. For a team, point it at Postgres: `DefaultDBConnector(database_url="postgresql://user:pw@host/db")`. (Snowflake has its own connector, `trulens-connectors-snowflake`.)

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, re, time, math
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from datasheet_rag.clients import oai                 # OpenAI client (loads .env, OS cert store)
from datasheet_rag.sparse import tokenize             # simple tokenizer for our toy retriever

from trulens.core import TruSession, Metric, Selector
from trulens.core.database.connector.default import DefaultDBConnector
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes
from trulens.apps.app import TruApp

Path("evals").mkdir(exist_ok=True)
session = TruSession(connector=DefaultDBConnector(database_url="sqlite:///evals/learning.sqlite"))
session.reset_database()          # start clean every time you run the course
APP_MODEL = "gpt-5.4-mini"        # model used by the toy apps

Database does not need migration.


## 1.2 The application under test

A deliberately small RAG app over LTC3310S datasheet facts:
- `retrieve()`: keyword-overlap retriever (top-k);
- `generate()`: one LLM call;
- `query()`: the entry point.

Two knobs create **versions** we'll compare later: `top_k`, and `style`. With `style="loose"` the prompt allows outside knowledge, which tends to cause hallucinations.

The `@instrument` decorators are all TruLens needs. Each decorated call becomes a span, and the **span type** plus **attributes** tell the metrics where to find query, contexts, and answer.

In [2]:
CORPUS = [
    "LTC3310S: 5V, 10A synchronous step-down Silent Switcher 2 regulator in a 3mm x 3mm LQFN package; input supply 2.25V to 5.5V.",
    "Electrical characteristics: regulated feedback voltage VFB is 495 mV min, 500 mV typ, 505 mV max over the full temperature range.",
    "Electrical characteristics: VIN undervoltage lockout (VIN rising) 2.0 V min, 2.1 V typ, 2.2 V max; UVLO hysteresis 150 mV typical.",
    "Electrical characteristics: top switch current limit 14.5 A min, 16 A typ, 18.5 A max (VOUT/VIN <= 0.2). Bottom switch current limit 10 A min, 12 A typ, 14 A max.",
    "Electrical characteristics: top switch (PMOS) on-resistance 16 mOhm typical; bottom switch (NMOS) on-resistance 4.5 mOhm typical.",
    "Oscillator: default switching frequency with RT tied to VIN is 2 MHz typ (1.8 MHz to 2.2 MHz). Programmable frequency range 0.5 MHz to 5 MHz.",
    "Setting the switching frequency: RT = 568 * fSW^(-1.08) with RT in kOhm and fSW in MHz. Table 1: 1 MHz -> 549 kOhm, 2 MHz -> 274 kOhm.",
    "Inductor selection: a good first choice is L >= VOUT/(3A * fSW) * (1 - VOUT/VIN(MAX)) for VOUT/VIN(MAX) <= 0.5, with fSW in MHz and L in uH.",
    "PGOOD pin: open-drain power good output of an internal comparator; pulled low when the output is out of regulation or the part is in shutdown. Rising threshold 98% of VOUT typ.",
    "Absolute maximum ratings: VIN -0.3 V to 6 V; maximum reflow (package body) temperature 260 C. These are stress ratings, not operating conditions.",
    "Variants: LTC3310S uses Silent Switcher 2 (SS2); LTC3310 uses Silent Switcher 1 (SS1); the -1 versions have a fixed 1 V output.",
]

class MiniRAG:
    def __init__(self, top_k: int = 2, style: str = "grounded", model: str = APP_MODEL):
        self.top_k, self.style, self.model = top_k, style, model

    @instrument(span_type=SpanAttributes.SpanType.RETRIEVAL,
                attributes={SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
                            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return"})
    def retrieve(self, query: str) -> list[str]:
        q = set(tokenize(query))
        scored = sorted(CORPUS, key=lambda doc: -len(q & set(tokenize(doc))))
        return scored[: self.top_k]

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def generate(self, query: str, contexts: list[str]) -> str:
        rules = ("Answer ONLY from the context. If the context does not contain the answer, say you don't know."
                 if self.style == "grounded" else
                 "Answer the question helpfully. Use the context and your general knowledge.")
        ctx = "\n".join(f"- {c}" for c in contexts)
        r = oai.responses.create(model=self.model, reasoning={"effort": "low"},
                                 instructions=f"You are a datasheet assistant. {rules} Be concise.",
                                 input=f"Context:\n{ctx}\n\nQuestion: {query}")
        return r.output_text

    @instrument(span_type=SpanAttributes.SpanType.RECORD_ROOT,
                attributes={SpanAttributes.RECORD_ROOT.INPUT: "query",
                            SpanAttributes.RECORD_ROOT.OUTPUT: "return"})
    def query(self, query: str) -> str:
        return self.generate(query, self.retrieve(query))

rag = MiniRAG()
print(rag.query("What is the UVLO hysteresis?"))      # plain call: works normally, nothing is recorded

Cannot explicitly set 'record_root' span type, setting to 'unknown'.


The UVLO hysteresis is **150 mV typical**.


## 1.3 Recording with `TruApp`

`TruApp(app, app_name, app_version, main_method, feedbacks)` registers the app. Calls made **inside `with tru_app:`** are recorded: a record root span plus child spans. The returned `recording` lets you fetch the record ids and feedback results.

Calls outside the `with` block are not recorded, so recording costs nothing in production paths you don't wrap.

In [3]:
tru_rag = TruApp(rag, app_name="mini-rag", app_version="v0-no-metrics", main_method=rag.query)

with tru_rag as recording:
    rag.query("What is the default switching frequency?")
    rag.query("What is the absolute maximum VIN?")

session.force_flush()                       # spans are exported in batches; flush before reading
records, feedback_cols = session.get_records_and_feedback()
display(records[["app_version", "input", "output", "latency", "total_tokens"]])

,app_version,input,output,latency,total_tokens
0,v0-no-metrics,What is the default switching frequency?,The default switching frequency is **2 MHz typ...,1.905185,0
1,v0-no-metrics,What is the absolute maximum VIN?,The absolute maximum VIN is **6 V**.,1.724615,0


## 1.4 Providers: the judge

A **provider** wraps an LLM (or classifier) and offers dozens of evaluation methods: `relevance`, `context_relevance`, `groundedness_measure_with_cot_reasons`, and so on. The official provider packages are:

| Package | Judge models |
|---|---|
| `trulens-providers-openai` | OpenAI / Azure OpenAI |
| `trulens-providers-litellm` | 100+ models through LiteLLM (Anthropic, Gemini, Ollama, …) |
| `trulens-providers-bedrock` | AWS Bedrock |
| `trulens-providers-cortex` | Snowflake Cortex |
| `trulens-providers-huggingface` | Classifier-based metrics (language match, groundedness NLI, …) |
| `trulens-providers-langchain` | Any LangChain chat model |

**Custom providers.** Any class implementing `LLMProvider._create_chat_completion` becomes a full judge. This project uses one (`datasheet_rag/trulens_judge.py`) because `trulens-providers-openai` pins `openai<2`, which conflicts with the Agent Framework stack. It calls the OpenAI Responses API with structured outputs. Read that file: it is about 20 lines.

`DummyProvider` returns random scores without calling any model. Use it to test plumbing for free.

In [4]:
from datasheet_rag.trulens_judge import ResponsesJudge
from trulens.feedback.dummy.provider import DummyProvider

judge = ResponsesJudge(model_engine="gpt-5.4-mini", reasoning_effort="low")
dummy = DummyProvider(delay=0.0, error_prob=0.0, loading_prob=0.0, freeze_prob=0.0, overloaded_prob=0.0)

score, reasons = judge.relevance_with_cot_reasons(prompt="What is VFB?", response="VFB is 500 mV typical.")
print("judge  relevance:", score, "\n", reasons["reason"][:400])
print("dummy  relevance:", dummy.relevance(prompt="What is VFB?", response="VFB is 500 mV typical."))

Package jsonschema not present in requirements.


Rating must be in [0, 3].


Multiple valid rating values found in the string: 
First an integer: 2 . Also, this is my response to a prompt of length
1600 with a model dummymodel with temperature 0.0. Also, here is
an integer in case this is being used as a score: 2



judge  relevance: 0.6666666666666666 
 Criteria: Score how relevant the response is to the prompt: it should address the full question to get 3, provide partial but related information for a lower score, and get 0 if it is irrelevant or confidently false.
Supporting Evidence: The prompt asks, “What is VFB?” The response, “VFB is 500 mV typical,” is related because it refers to VFB and gives a typical value, so it is not irrelevant. How
dummy  relevance: 0.0


## 1.5 The RAG triad: your first metrics

The three questions that catch most RAG failures:

| Metric | Question | Selectors |
|---|---|---|
| **Context relevance** | Did retrieval return useful chunks? | `question` = record input, `context` = each retrieved context (`collect_list=False`), then aggregated with `mean` |
| **Groundedness** | Is the answer supported by the contexts? (hallucination) | `source` = all contexts (`collect_list=True`), `statement` = record output |
| **Answer relevance** | Does the answer address the question? | input and output (`on_input_output()`) |

Scores are normalized to **[0, 1]**. The `_with_cot_reasons` variants also return the judge's chain-of-thought explanation. Prefer them: a score you can't explain can't be debugged.

In [5]:
f_answer_rel = Metric(judge.relevance_with_cot_reasons, name="Answer relevance").on_input_output()

f_context_rel = (Metric(judge.context_relevance_with_cot_reasons, name="Context relevance")
                 .on({"question": Selector.select_record_input(),
                      "context": Selector.select_context(collect_list=False)})
                 .aggregate(np.mean))

f_groundedness = (Metric(judge.groundedness_measure_with_cot_reasons, name="Groundedness")
                  .on({"source": Selector.select_context(collect_list=True),
                       "statement": Selector.select_record_output()}))

RAG_TRIAD = [f_answer_rel, f_context_rel, f_groundedness]

QUESTIONS = [
    "What is the VIN undervoltage lockout hysteresis?",
    "What RT resistor gives 1 MHz?",
    "What is the top switch current limit?",
    "What does the PGOOD pin do?",
    "What is the thermal resistance theta-JA of the package?",     # NOT in the corpus -> tests honesty
]

def run_app(app_obj, version: str, metrics: list, questions=QUESTIONS, app_name="mini-rag"):
    tru = TruApp(app_obj, app_name=app_name, app_version=version, main_method=app_obj.query, feedbacks=metrics)
    with tru as rec:
        for q in questions:
            app_obj.query(q)
    return tru, rec

tru_v1, rec_v1 = run_app(MiniRAG(top_k=2, style="grounded"), "v1-top2-grounded", RAG_TRIAD)
results_v1 = rec_v1.retrieve_feedback_results(timeout=300)   # waits until the metrics for these records are done
display(results_v1)

,Answer relevance,Context relevance,Groundedness
record_id,,,
cda14361-8783-44de-9ab2-fa5a3393cf9e,1.0,0.500000,1.0
be4cf7b8-de3a-4e69-9482-98d48e37d752,1.0,0.833333,1.0
25a64bc5-31d9-415d-bb0d-bd184e63991e,1.0,0.666667,1.0
48cae832-0cf5-4b65-8009-c488e62f01e4,1.0,0.666667,1.0
59c6ba62-1093-45c5-913b-4c132b9fb55b,0.0,0.333333,0.0


## 1.6 Reading results

| API | Returns |
|---|---|
| `session.get_leaderboard()` | Mean score per metric, latency, and cost per `app_name`/`app_version` |
| `session.get_records_and_feedback()` | One row per record: input, output, latency, tokens, cost, one column per metric, plus `<metric>_calls` with per-call details |
| `recording.retrieve_feedback_results()` | Results for the records of one `with` block |
| `session.get_events(app_name, app_version)` | The raw OTel spans, including EVAL spans with each score's **explanation** |

The helper below pulls the judge explanations out of the EVAL spans. This is the most useful debugging view.

In [6]:
display(session.get_leaderboard())

EV = SpanAttributes.EVAL_ROOT
def explanations(app_name: str, app_version: str, metric: str | None = None, max_chars: int = 500) -> pd.DataFrame:
    ev = session.get_events(app_name=app_name, app_version=app_version)
    rows = []
    for _, e in ev.iterrows():
        a = e["record_attributes"]
        if a.get(SpanAttributes.SPAN_TYPE) != "eval_root":
            continue
        if metric and a.get(EV.METRIC_NAME) != metric:
            continue
        rows.append({"metric": a.get(EV.METRIC_NAME), "score": a.get(EV.SCORE),
                     "record_id": a.get(SpanAttributes.RECORD_ID),
                     "explanation": str(a.get(EV.EXPLANATION, ""))[:max_chars]})
    return pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 300)
display(explanations("mini-rag", "v1-top2-grounded", "Groundedness"))

Answer relevance  Context relevance  Groundedness  \
app_name app_version                                                           
mini-rag v1-top2-grounded               0.8                0.6           0.8   
         v0-no-metrics                  NaN                NaN           NaN   

                           latency  total_cost  
app_name app_version                            
mini-rag v1-top2-grounded  1.73236         0.0  
         v0-no-metrics     1.81490         0.0

,metric,score,record_id,explanation
0,Groundedness,1.0,25a64bc5-31d9-415d-bb0d-bd184e63991e,
1,Groundedness,1.0,48cae832-0cf5-4b65-8009-c488e62f01e4,
2,Groundedness,0.0,59c6ba62-1093-45c5-913b-4c132b9fb55b,
3,Groundedness,1.0,be4cf7b8-de3a-4e69-9482-98d48e37d752,
4,Groundedness,1.0,cda14361-8783-44de-9ab2-fa5a3393cf9e,


## 1.7 The dashboard

`run_dashboard(session)` starts a Streamlit UI with these tabs:
- **Leaderboard:** versions × metrics.
- **Records:** per-record trace viewer, with each metric's reasoning.
- **Compare:** side-by-side records from different versions.
- **Trends:** metrics over time.
- **Prompts:** the prompt registry (§3.9).

From a terminal (this project): `uv run python trulens_dashboard.py` and point it at `evals/learning.sqlite`, or run the cell below with `RUN_DASHBOARD = True`. On Windows, `force=True` (restart an existing dashboard) isn't supported, so stop the old one first.

In [7]:
RUN_DASHBOARD = False
if RUN_DASHBOARD:
    from trulens.dashboard import run_dashboard
    run_dashboard(session, port=8502, address="127.0.0.1")    # http://localhost:8502

# Part 2: Intermediate

## 2.1 Instrumentation in depth

`@instrument(name=None, span_type=SpanType.UNKNOWN, attributes=None)`

- **span_type** marks semantics: `RECORD_ROOT`, `RETRIEVAL`, `GENERATION`, `RERANKER`, `TOOL`, `AGENT`, `WORKFLOW_STEP`, `GRAPH_NODE`/`GRAPH_TASK`, `MCP`, `GUARDRAIL`, and so on. Standard selectors such as `select_context()` find data by span type.
- **attributes** maps attribute keys to:
  - a **parameter name** of the function (`"query"`), or `"return"` for the return value;
  - a **callable** `(ret, exception, *args, **kwargs) -> dict` for computed attributes. It runs after the call, so it can read the result.
- Every instrumented call also records `ai.observability.call.function`, `.kwargs`, `.return`, and `.error` automatically.
- Works on sync and async functions and on generators.
- If you don't own the code, you can instrument it without decorators: `instrument.instrument_method(cls, "method_name", span_type=…)`, or the framework recorders `TruChain` (LangChain), `TruLlama` (LlamaIndex) and `TruGraph` (LangGraph), which instrument automatically.

Below, a re-ranker step with a **computed attribute**: the number of contexts dropped. A custom metric then reads it.

In [8]:
class MiniRAGRerank(MiniRAG):
    @instrument(span_type=SpanAttributes.SpanType.RERANKER,
                attributes=lambda ret, exception, *args, **kwargs: {
                    "mini_rag.rerank.kept": len(ret or []),
                    "mini_rag.rerank.dropped": len(kwargs.get("contexts", args[1] if len(args) > 1 else [])) - len(ret or []),
                })
    def rerank(self, query: str, contexts: list[str]) -> list[str]:
        q = set(tokenize(query))
        return [c for c in contexts if len(q & set(tokenize(c))) >= 2] or contexts[:1]

    @instrument(span_type=SpanAttributes.SpanType.RECORD_ROOT,
                attributes={SpanAttributes.RECORD_ROOT.INPUT: "query", SpanAttributes.RECORD_ROOT.OUTPUT: "return"})
    def query(self, query: str) -> str:
        ctx = self.rerank(query, self.retrieve(query))
        return self.generate(query, ctx)

# a metric that reads the custom span attribute directly
def kept_ratio(kept: int) -> float:
    return min(1.0, kept / 2)

f_kept = Metric(kept_ratio, name="Rerank kept ratio").on(
    {"kept": Selector(span_type=SpanAttributes.SpanType.RERANKER, span_attribute="mini_rag.rerank.kept")})

tru_rr, rec_rr = run_app(MiniRAGRerank(top_k=4), "v1b-rerank", [f_kept], QUESTIONS[:2])
display(rec_rr.retrieve_feedback_results(timeout=120))

Cannot explicitly set 'record_root' span type, setting to 'unknown'.


Metric implementation <function kept_ratio at 0x00000213AD3952D0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


,Rerank kept ratio
record_id,
3654e99d-c7ca-4b56-ba85-e3ff8d77b260,0.5
a9ad7f95-1d5b-405e-b16e-71784164bd70,1.0


## 2.2 Selectors: wiring trace data into metric arguments

A metric's implementation has named parameters, and each is bound to a `Selector`:

| Selector | Picks |
|---|---|
| `Selector.select_record_input()` / `select_record_output()` | Record root input / output |
| `Selector.select_context(collect_list=…)` | Retrieved contexts (RETRIEVAL spans) |
| `Selector(span_type=…, span_attribute=…)` | Any attribute of spans of a given type |
| `Selector(function_name="pkg.Class.method", function_attribute="return" \| "<arg>")` | Return value or argument of a specific instrumented function |
| `Selector(span_name=…)` | Spans by name |
| `Selector(span_attributes_processor=lambda attrs: …)` | Arbitrary transform of a span's attributes |
| `Selector(trace_level=True)` | The **whole trace** (agent metrics, §3.1) |
| `Selector(conversation_level=True)` / `.on_conversation()` | All records of a conversation (§3.2) |
| `Selector.from_column("col")` | A dataset column (batch evals, §3.4) |

Key options:
- `collect_list=True` passes the whole list once. `False` calls the metric per element; combine it with `.aggregate(fn)`.
- `ignore_none_values=True` skips the evaluation when the data is missing.
- `match_only_if_no_ancestor_matched=True` avoids double counting for nested spans of the same type.

The shortcuts `on_input()`, `on_output()`, `on_input_output()`, `on_context(collect_list=…)` and `on_default()` cover the common cases.

In [9]:
# Same metric, two different bindings: per-context (aggregated) vs whole list at once
f_ctx_each = Metric(judge.context_relevance, name="Ctx relevance (per chunk, min)").on(
    {"question": Selector.select_record_input(), "context": Selector.select_context(collect_list=False)}).aggregate(np.min)

def n_contexts(contexts: list) -> float:
    return float(len(contexts))
f_ctx_count = Metric(n_contexts, name="Num contexts", higher_is_better=False).on(
    {"contexts": Selector.select_context(collect_list=True)})

# select a specific function's return value by its qualified name
f_gen_len = Metric(lambda text: min(1.0, len(text) / 400), name="Answer length (norm)", higher_is_better=False).on(
    {"text": Selector(function_name=f"{MiniRAG.__module__}.{MiniRAG.generate.__qualname__}", function_attribute="return")})

_, rec_sel = run_app(MiniRAG(top_k=3), "v1c-selectors", [f_ctx_each, f_ctx_count, f_gen_len], QUESTIONS[:2])
display(rec_sel.retrieve_feedback_results(timeout=120))

Metric implementation <function n_contexts at 0x00000213AD396560> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


Metric implementation <function <lambda> at 0x00000213AD397270> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


,"Ctx relevance (per chunk, min)",Num contexts,Answer length (norm)
record_id,,,
cdeccbc8-60e2-46d9-95b9-bc1105bb7daf,0.000000,3.0,0.1550
8ab27662-2944-459a-b901-d7ee83e1183c,0.333333,3.0,0.0875


## 2.3 The metric catalog (LLM-judge methods on every `LLMProvider`)

| Family | Methods (most have `_with_cot_reasons`) | Typical selectors |
|---|---|---|
| **RAG** | `context_relevance`, `relevance` (answer relevance), `groundedness_measure_with_cot_reasons`, `groundedness_measure_with_cot_reasons_consider_answerability`, `comprehensiveness_with_cot_reasons`, `summarization_with_cot_reasons` | input / contexts / output |
| **Correctness** | `correctness`, `GroundTruthAgreement.agreement_measure` (§2.6) | output (+ ground truth) |
| **Citations** | `citation_accuracy`, `citation_attribution` | response + contexts |
| **Quality / style** | `coherence`, `conciseness`, `helpfulness`, `sentiment`, `controversiality` | output |
| **Safety** | `harmfulness`, `maliciousness`, `criminality`, `insensitivity`, `misogyny`, `stereotypes` | output (`higher_is_better=False`) |
| **Requirements** | `requirement_satisfaction_with_cot_reasons`, `topic_adherence` | request + output |
| **Agents (trace-level)** | `plan_quality`, `plan_adherence`, `tool_selection`, `tool_calling`, `tool_quality`, `logical_consistency`, `execution_efficiency`, `agent_goal_accuracy` | `Selector(trace_level=True)` |
| **Conversations** | `coherence_across_turns`, `conversation_helpfulness` | `.on_conversation()` |
| **Building blocks** | `generate_score`, `generate_score_and_reasons` | anything (custom judges, §2.4) |

The list below is generated from the installed version, so it is always current:

In [10]:
from trulens.feedback.llm_provider import LLMProvider
skip = {"generate_score", "generate_score_and_reasons", "get_class", "of_class", "of_object", "replace", "update",
        "model_agreement", "clear_capabilities_cache", "clear_model_capabilities_cache"}
catalog = sorted(m for m in dir(LLMProvider) if not m.startswith(("_", "model")) and callable(getattr(LLMProvider, m))
                 and m not in skip and m not in dir(__import__("pydantic").BaseModel))
print(len(catalog), "judge methods:\n", ", ".join(catalog))

55 judge methods:
 agent_goal_accuracy, agent_goal_accuracy_with_cot_reasons, citation_accuracy, citation_accuracy_with_cot_reasons, citation_attribution, citation_attribution_with_cot_reasons, coherence, coherence_across_turns, coherence_across_turns_with_cot_reasons, coherence_with_cot_reasons, comprehensiveness_with_cot_reasons, conciseness, conciseness_with_cot_reasons, context_relevance, context_relevance_with_cot_reasons, controversiality, controversiality_with_cot_reasons, conversation_helpfulness, conversation_helpfulness_with_cot_reasons, correctness, correctness_with_cot_reasons, criminality, criminality_with_cot_reasons, execution_efficiency_with_cot_reasons, groundedness_measure_with_cot_reasons, groundedness_measure_with_cot_reasons_consider_answerability, harmfulness, harmfulness_with_cot_reasons, helpfulness, helpfulness_with_cot_reasons, insensitivity, insensitivity_with_cot_reasons, load, logical_consistency_with_cot_reasons, maliciousness, maliciousness_with_cot_reaso

In [11]:
# A few non-RAG metrics on the same app; safety metrics are 'lower is better'
QUALITY = [
    Metric(judge.coherence_with_cot_reasons, name="Coherence").on_output(),
    Metric(judge.conciseness_with_cot_reasons, name="Conciseness").on_output(),
    Metric(judge.harmfulness_with_cot_reasons, name="Harmfulness", higher_is_better=False).on_output(),
    Metric(judge.comprehensiveness_with_cot_reasons, name="Comprehensiveness").on(
        {"source": Selector.select_context(collect_list=True), "summary": Selector.select_record_output()}),
]
_, rec_q = run_app(MiniRAG(top_k=2), "v1d-quality", QUALITY, QUESTIONS[:2])
display(rec_q.retrieve_feedback_results(timeout=180))

Multiple valid rating values found in the string: Score: 0, where 0 is the lowest score according to the criteria and 3 is the highest possible score


,Coherence,Conciseness,Harmfulness,Comprehensiveness
record_id,,,,
dd94f029-78fe-4fe6-8c03-3de7bc2d8fc5,1.0,1.0,0.0,0.111111
0ffb9f12-c140-4879-a085-a3cfabb64f73,1.0,1.0,0.0,0.166667


## 2.4 Customizing LLM judges

Every rubric-based method accepts:
- `criteria=` replaces the default rubric with your own definition of "good";
- `additional_instructions=` appends domain guidance without replacing the rubric;
- `examples=` few-shot calibration examples;
- `min_score_val` / `max_score_val` set the scale the judge uses (0–3 by default). The result is still normalized to 0–1.

These can be passed through `Metric(...)` so they stay attached to the metric definition. For **fully custom judges**, write a function that calls `provider.generate_score_and_reasons(system_prompt, user_prompt, min_score_val, max_score_val)` (the method that powers all the built-ins).

In [12]:
DOMAIN = ("This is a semiconductor datasheet assistant. A good answer states numeric values with units and says whether "
          "they are MIN/TYP/MAX and under which conditions. Penalize answers that omit units or mix typical and guaranteed values.")

f_rel_domain = Metric(judge.relevance_with_cot_reasons, name="Answer relevance (domain)",
                      additional_instructions=DOMAIN).on_input_output()

f_correct_strict = Metric(judge.correctness_with_cot_reasons, name="Correctness (strict criteria)",
                          criteria="Score high only if every number in the response is plausible for the question AND carries "
                                   "units AND indicates whether it is typical or guaranteed.",
                          min_score_val=0, max_score_val=10).on_output()

def spec_completeness(question: str, answer: str) -> tuple[float, dict]:
    """Fully custom judge: does the answer give a spec in engineering-complete form?"""
    system = ("You grade answers from a datasheet assistant. Score 0-4: +1 numeric value present, +1 units present, "
              "+1 states MIN/TYP/MAX type, +1 states conditions. If the answer correctly says the info is unavailable, give 4.")
    return judge.generate_score_and_reasons(system_prompt=system, user_prompt=f"QUESTION: {question}\nANSWER: {answer}",
                                            min_score_val=0, max_score_val=4)

f_spec = Metric(spec_completeness, name="Spec completeness").on_input_output()

_, rec_custom = run_app(MiniRAG(top_k=2), "v1e-custom-judges", [f_rel_domain, f_correct_strict, f_spec], QUESTIONS[:3])
display(rec_custom.retrieve_feedback_results(timeout=180))
display(explanations("mini-rag", "v1e-custom-judges", "Spec completeness"))

Metric implementation <function spec_completeness at 0x00000213ACF81010> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


,Answer relevance (domain),Correctness (strict criteria),Spec completeness
record_id,,,
0c4ddcf3-b274-4ae0-bae5-4400258cfbdd,1.000000,1.0,0.75
8f53df44-fe91-4b32-9c49-ed2345536c67,0.666667,0.4,0.50
972be4ed-b897-4bf0-a121-5d2420355f7d,1.000000,1.0,0.75


,metric,score,record_id,explanation
0,Spec completeness,0.75,0c4ddcf3-b274-4ae0-bae5-4400258cfbdd,
1,Spec completeness,0.50,8f53df44-fe91-4b32-9c49-ed2345536c67,
2,Spec completeness,0.75,972be4ed-b897-4bf0-a121-5d2420355f7d,


## 2.5 Deterministic (non-LLM) metrics

Plain Python functions are metrics too. They are fast, free, reproducible, and ideal for hard requirements:
- format checks;
- regex rules: has a citation, has units, no forbidden phrases;
- JSON or schema validity: `SchemaValidator(schema=PydanticModel or JSON schema).validate_json`;
- length and latency budgets;
- exact or numeric match against expected values.

Return a float in [0, 1], or `(score, {"reason": …})` to attach an explanation.

In [13]:
UNIT = re.compile(r"\b\d+(\.\d+)?\s*(m?V|mA|A|µA|uA|mΩ|mOhm|kΩ|kOhm|Ω|MHz|kHz|°C|C|%|µH|uH|ns)\b")

def has_units(answer: str) -> tuple[float, dict]:
    hits = UNIT.findall(answer)
    return (1.0 if hits else 0.0), {"reason": f"{len(hits)} value(s) with units found"}

def admits_unknown(question: str, answer: str) -> float:
    """1.0 if the answer admits it doesn't know (use with questions you KNOW are unanswerable)."""
    return 1.0 if re.search(r"don.t know|not (provided|available|in the context|specified)|cannot", answer, re.I) else 0.0

from pydantic import BaseModel
from trulens.feedback import SchemaValidator
class SpecJSON(BaseModel):
    parameter: str
    value: float
    unit: str
validator = SchemaValidator(schema=SpecJSON)
print("schema check:", validator.validate_json('{"parameter": "VFB", "value": 500, "unit": "mV"}'),
      validator.validate_json('{"parameter": "VFB"}')[0])

DETERMINISTIC = [Metric(has_units, name="Has units").on_output(),
                 Metric(admits_unknown, name="Admits unknown").on_input_output()]
_, rec_det = run_app(MiniRAG(top_k=2), "v1f-deterministic", DETERMINISTIC)
display(rec_det.retrieve_feedback_results(timeout=120))

Metric implementation <function has_units at 0x00000213ACF828D0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


Metric implementation <function admits_unknown at 0x00000213ACF83B60> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


schema check: (1.0, {'explanation': 'Output conforms to the schema.'}) 0.0


,Has units,Admits unknown
record_id,,
7980156b-cc25-4af5-9306-acaf2954d732,1.0,0.0
95fee652-a839-4932-b4c3-deef37cd700d,1.0,0.0
9ba9cf6e-4682-476f-a81b-cbba8e0c865a,1.0,0.0
de904570-0a86-4a31-84b8-48e7ce91cfab,0.0,0.0
1e47b16c-ff25-44cd-b252-e53d612c00fc,0.0,1.0


## 2.6 Ground truth: correctness against expected answers

There are three ways to bring in ground truth:
1. **`GroundTruthAgreement(golden, provider)`**: an LLM judge compares the answer with the expected response, looked up by the record's input.
2. **Datasets stored in TruLens:** `session.add_ground_truth_to_dataset(name, df)` and `session.get_ground_truth(name)`. They are versioned with your evals and shared across the team.
3. **Retrieval ground truth:** compare retrieved chunks with `expected_chunks` (precision, recall, …). `GroundTruthAggregator` offers ranking and classification aggregations (`precision`, `recall`, `f1_score`, `auc`, `pearson_correlation`, `cohens_kappa`, `brier_score`, `ece`, …).

In [14]:
from trulens.feedback import GroundTruthAgreement

# expected_chunks is a list of dicts (e.g. {"text": ..., "title": ...}) in the TruLens GroundTruth schema
golden_df = pd.DataFrame([
    {"query": QUESTIONS[0], "expected_response": "UVLO hysteresis is 150 mV typical.", "expected_chunks": [{"text": CORPUS[2]}]},
    {"query": QUESTIONS[1], "expected_response": "RT = 549 kOhm for 1 MHz.",             "expected_chunks": [{"text": CORPUS[6]}]},
    {"query": QUESTIONS[2], "expected_response": "Top switch current limit: 14.5 A min, 16 A typ, 18.5 A max.", "expected_chunks": [{"text": CORPUS[3]}]},
    {"query": QUESTIONS[3], "expected_response": "PGOOD is an open-drain power-good output, pulled low when out of regulation or in shutdown.", "expected_chunks": [{"text": CORPUS[8]}]},
    {"query": QUESTIONS[4], "expected_response": "The information is not available.", "expected_chunks": []},
])
session.add_ground_truth_to_dataset(dataset_name="mini_rag_golden", ground_truth_df=golden_df,
                                    dataset_metadata={"domain": "LTC3310S", "owner": "eval-course"})
gt = session.get_ground_truth("mini_rag_golden")
display(gt[["query", "expected_response"]])

gta = GroundTruthAgreement(gt, provider=judge)
f_correct = Metric(gta.agreement_measure, name="Correctness (vs golden)").on_input_output()

# deterministic retrieval recall against expected chunks
EXPECTED = {r["query"]: {c["text"] for c in r["expected_chunks"]} for r in golden_df.to_dict("records")}
def retrieval_recall(question: str, contexts: list) -> float:
    exp = EXPECTED.get(question, set())
    return 1.0 if not exp else len(exp & set(contexts)) / len(exp)
f_recall = Metric(retrieval_recall, name="Retrieval recall").on(
    {"question": Selector.select_record_input(), "contexts": Selector.select_context(collect_list=True)})

GOLDEN_METRICS = [f_correct, f_recall]

,query,expected_response
0,What is the VIN undervoltage lockout hysteresis?,UVLO hysteresis is 150 mV typical.
1,What RT resistor gives 1 MHz?,RT = 549 kOhm for 1 MHz.
2,What is the top switch current limit?,"Top switch current limit: 14.5 A min, 16 A typ, 18.5 A max."
3,What does the PGOOD pin do?,"PGOOD is an open-drain power-good output, pulled low when out of regulation or in shutdown."
4,What is the thermal resistance theta-JA of the package?,The information is not available.


Metric implementation <function retrieval_recall at 0x00000213ACF81430> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


## 2.7 Groundedness in depth

`groundedness_measure_with_cot_reasons(source, statement)`:
1. Splits the answer into statements (`GroundednessConfigs(use_sent_tokenize=True)`).
2. Optionally drops trivial statements such as "Sure!" (`filter_trivial_statements=True`).
3. Asks the judge whether each statement is supported by the source.
4. Averages the results.

The `_consider_answerability` variant gives full credit when the source lacks the answer **and** the response says so. That is what you want for "I don't know" behaviour.

In [15]:
from trulens.core.metric.metric import GroundednessConfigs

f_ground_cfg = Metric(judge.groundedness_measure_with_cot_reasons, name="Groundedness (filtered)",
                      groundedness_configs=GroundednessConfigs(use_sent_tokenize=True, filter_trivial_statements=True)
                      ).on({"source": Selector.select_context(collect_list=True), "statement": Selector.select_record_output()})

f_ground_ans = Metric(judge.groundedness_measure_with_cot_reasons_consider_answerability,
                      name="Groundedness (answerability)").on(
    {"source": Selector.select_context(collect_list=True), "statement": Selector.select_record_output(),
     "question": Selector.select_record_input()})

GROUNDED_VARIANTS = [f_ground_cfg, f_ground_ans]

## 2.8 Versions and experiments

Change one thing per version, keep the question set fixed, and compare on the leaderboard. Below, three versions of the app run on the same metrics:
- **v2-top1-grounded:** less context.
- **v2-top3-grounded:** more context.
- **v2-top3-loose:** the prompt allows outside knowledge. Expect lower groundedness, and inventing an answer to the unanswerable θJA question.

In [16]:
SUITE = RAG_TRIAD + GOLDEN_METRICS + GROUNDED_VARIANTS + DETERMINISTIC
VERSIONS = {"v2-top1-grounded": MiniRAG(top_k=1, style="grounded"),
            "v2-top3-grounded": MiniRAG(top_k=3, style="grounded"),
            "v2-top3-loose":    MiniRAG(top_k=3, style="loose")}
recs = {}
for version, app_obj in VERSIONS.items():
    _, recs[version] = run_app(app_obj, version, SUITE)
for version, rec in recs.items():
    rec.retrieve_feedback_results(timeout=600)

lb = session.get_leaderboard()
display(lb[lb.index.get_level_values("app_version").str.startswith("v2")].T.round(2)
        if isinstance(lb.index, pd.MultiIndex) else lb.round(2))

app_name                               mini-rag                               
app_version                    v2-top1-grounded v2-top3-grounded v2-top3-loose
Admits unknown                             0.40             0.20          0.00
Answer length (norm)                        NaN              NaN           NaN
Answer relevance                           0.60             0.80          1.00
Answer relevance (domain)                   NaN              NaN           NaN
Coherence                                   NaN              NaN           NaN
Comprehensiveness                           NaN              NaN           NaN
Conciseness                                 NaN              NaN           NaN
Context relevance                          0.80             0.49          0.51
Correctness (strict criteria)               NaN              NaN           NaN
Correctness (vs golden)                    0.80             0.98          0.82
Ctx relevance (per chunk, min)              NaN              NaN           NaN
Groundedness                               0.73             0.80          0.80
Groundedness (answerability)               0.80             0.80          0.70
Groundedness (filtered)                    0.80             0.80          0.80
Harmfulness                                 NaN              NaN           NaN
Has units                                  0.60             0.60          0.80
Num contexts                                NaN              NaN           NaN
Rerank kept ratio                           NaN              NaN           NaN
Retrieval recall                           0.80             1.00          1.00
Spec completeness                           NaN              NaN           NaN
latency                                    1.95             1.99          2.51
total_cost                                 0.00             0.00          0.00

# Part 3: Advanced

## 3.1 Agent evaluation: trace-level metrics

An answer-level score can't tell you *why* an agent failed. TruLens's agent metrics read the **whole trace** (`Selector(trace_level=True)`): every LLM step, tool call, argument and result. They then judge the process:

| Metric | Asks |
|---|---|
| `plan_quality` | Was the plan sensible and complete for the goal? |
| `plan_adherence` | Did execution follow the plan? |
| `tool_selection` | Were the right tools chosen? |
| `tool_calling` | Were the calls well-formed, with correct arguments? |
| `tool_quality` | Did the tools return useful results (tool-side failures)? |
| `logical_consistency` | Are the steps coherent, without contradictions? |
| `execution_efficiency` | No redundant or wasted steps? |
| `agent_goal_accuracy` | Was the user's goal achieved? |

Traces can be long, so `enable_trace_compression=True` (the default) compresses them before judging. Below is a small tool-using agent: an LLM with function calling, a spec lookup, and a calculator. Each step is instrumented as an AGENT, GENERATION or TOOL span.

In [17]:
TOOLS_SCHEMA = [
    {"type": "function", "name": "lookup_spec", "description": "Search the datasheet facts for a parameter.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"], "additionalProperties": False},
     "strict": True},
    {"type": "function", "name": "calculate", "description": "Evaluate an arithmetic expression.",
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"], "additionalProperties": False},
     "strict": True},
]

class MiniAgent:
    def __init__(self, model: str = APP_MODEL, max_steps: int = 6):
        self.model, self.max_steps = model, max_steps

    @instrument(span_type=SpanAttributes.SpanType.TOOL)
    def lookup_spec(self, query: str) -> str:
        q = set(tokenize(query))
        return "\n".join(sorted(CORPUS, key=lambda d: -len(q & set(tokenize(d))))[:2])

    @instrument(span_type=SpanAttributes.SpanType.TOOL)
    def calculate(self, expression: str) -> str:
        if not re.fullmatch(r"[0-9.\s+\-*/()^e]+", expression):
            return "error: only arithmetic allowed"
        return str(eval(expression.replace("^", "**"), {"__builtins__": {}}))   # input restricted by the regex above

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def think(self, messages: list) -> object:
        return oai.responses.create(model=self.model, reasoning={"effort": "low"}, tools=TOOLS_SCHEMA, input=messages,
                                    instructions="You answer datasheet design questions. First state a short plan, "
                                                 "then use tools. Use calculate for every arithmetic step.")

    @instrument(span_type=SpanAttributes.SpanType.AGENT,
                attributes={SpanAttributes.RECORD_ROOT.INPUT: "question", SpanAttributes.RECORD_ROOT.OUTPUT: "return"})
    def query(self, question: str) -> str:
        messages = [{"role": "user", "content": question}]
        for _ in range(self.max_steps):
            r = self.think(messages)
            calls = [o for o in r.output if o.type == "function_call"]
            if not calls:
                return r.output_text
            messages += r.output
            for c in calls:
                out = getattr(self, c.name)(**json.loads(c.arguments))
                messages.append({"type": "function_call_output", "call_id": c.call_id, "output": out})
        return "Stopped: step limit reached."

agent = MiniAgent()
AGENT_QUESTIONS = [
    "Which RT resistor gives 1 MHz, and what minimum inductor does the guideline give for 5 V in, 1.2 V out at 1 MHz?",
    "By how many millivolts can VFB deviate from its typical value, worst case?",
]
trace = Selector(trace_level=True)
AGENT_METRICS = [Metric(getattr(judge, f"{m}_with_cot_reasons"), name=m.replace("_", " ").title()).on({"trace": trace})
                 for m in ["plan_quality", "plan_adherence", "tool_selection", "tool_calling",
                           "logical_consistency", "execution_efficiency"]]
AGENT_METRICS.append(Metric(judge.relevance_with_cot_reasons, name="Answer relevance").on_input_output())

_, rec_agent = run_app(agent, "agent-v1", AGENT_METRICS, AGENT_QUESTIONS, app_name="mini-agent")
display(rec_agent.retrieve_feedback_results(timeout=600).T)
display(explanations("mini-agent", "agent-v1", "Execution Efficiency"))

record_id,4579c996-eb56-4c26-b8f0-aa1e2cd2b467,54fb14f0-a69d-4e91-a3bf-823477d96d8a
Plan Quality,0.000000,0.000000
Plan Adherence,0.666667,0.666667
Tool Selection,1.000000,1.000000
Tool Calling,1.000000,1.000000
Logical Consistency,0.666667,0.666667
Execution Efficiency,0.666667,0.333333
Answer relevance,1.000000,1.000000


,metric,score,record_id,explanation
0,Execution Efficiency,0.666667,4579c996-eb56-4c26-b8f0-aa1e2cd2b467,
1,Execution Efficiency,0.333333,54fb14f0-a69d-4e91-a3bf-823477d96d8a,


## 3.2 Multi-turn conversations

Pass `conversation_id` when entering the recorder. Every record in that block shares the id, and **conversation-level metrics** see all of them in order:
- `coherence_across_turns`
- `conversation_helpfulness`
- `requirement_satisfaction` over the whole thread

Bind them with `.on_conversation()`. Record-level metrics still run per turn. `tru_app.compute_feedbacks(metric_scope="conversation")` recomputes only conversation metrics.

In [18]:
class ChatRAG(MiniRAG):
    def __init__(self, **kw):
        super().__init__(**kw); self.history = []
    @instrument(span_type=SpanAttributes.SpanType.RECORD_ROOT,
                attributes={SpanAttributes.RECORD_ROOT.INPUT: "query", SpanAttributes.RECORD_ROOT.OUTPUT: "return"})
    def query(self, query: str) -> str:
        context = self.retrieve(query + " " + " ".join(h[0] for h in self.history[-2:]))
        convo = "".join(f"User: {q}\nAssistant: {a}\n" for q, a in self.history[-3:])
        answer = self.generate(f"{convo}User: {query}", context)
        self.history.append((query, answer))
        return answer

chat = ChatRAG(top_k=3)
CONVO_METRICS = [
    Metric(judge.coherence_across_turns, name="Coherence across turns").on_conversation(),
    Metric(judge.conversation_helpfulness_with_cot_reasons, name="Conversation helpfulness").on_conversation(),
    Metric(judge.relevance_with_cot_reasons, name="Answer relevance").on_input_output(),     # per turn
]
tru_chat = TruApp(chat, app_name="mini-chat", app_version="chat-v1", main_method=chat.query, feedbacks=CONVO_METRICS)
with tru_chat(conversation_id="conv-001") as rec_chat:
    for turn in ["What is the LTC3310S?", "What is its maximum input voltage?", "And how do I set it to 1 MHz?"]:
        print("USER:", turn, "\nBOT: ", chat.query(turn)[:200], "\n")
session.force_flush()
display(rec_chat.retrieve_feedback_results(timeout=300))
display(session.get_records_and_feedback()[0].query("app_name == 'mini-chat'")[["conversation_id", "input"]])

Cannot explicitly set 'record_root' span type, setting to 'unknown'.


USER: What is the LTC3310S? 
BOT:  The LTC3310S is a 5V, 10A synchronous step-down Silent Switcher 2 regulator in a 3 mm x 3 mm LQFN package, with a 2.25 V to 5.5 V input supply range. 



USER: What is its maximum input voltage? 
BOT:  Its maximum input voltage is 6 V absolute maximum. 



USER: And how do I set it to 1 MHz? 
BOT:  To set it to 1 MHz, use an RT resistor of **549 kΩ**. 



,Answer relevance,Coherence across turns,Conversation helpfulness
record_id,,,
96730633-878e-4059-b625-38ea90d8d3bc,1.0,NaN,NaN
4967ec8c-f3dc-4ab5-8fdb-38a59f662c15,1.0,NaN,NaN
f89cabc8-7185-49dd-9152-82c9d2ce9513,1.0,1.0,1.0


,conversation_id,input
38,conv-001,What is the LTC3310S?
39,conv-001,What is its maximum input voltage?
40,conv-001,And how do I set it to 1 MHz?


## 3.3 Evaluation modes and timing

| Mode | Behaviour | Use for |
|---|---|---|
| Default (OTel) | An evaluator thread computes metrics shortly after each record is flushed | Interactive work, notebooks |
| `tru_app.compute_feedbacks()` | Compute now, synchronously, over the app's events (`metric_scope="record"` / `"conversation"` / `"all"`) | Scripts, CI, recomputing after adding a metric |
| `session.compute_feedbacks_on_events(events_df, metrics)` | Evaluate spans already stored (e.g. from production), **with new metrics** | Back-testing a new metric on old traffic |
| `session.configure_online_eval(...)` | Sampled automatic evaluation (§3.6) | Production |
| `tru_app.stop_evaluator()` / `start_evaluator()` | Control the background evaluator | Avoid duplicate work |

Waiting helpers:
- `recording.retrieve_feedback_results(timeout=…)`
- `session.wait_for_records(record_ids)`
- `session.wait_for_feedback_results(record_ids, feedback_names)`

**Back-testing:** below we take records produced in §2.8 and score them with a metric that didn't exist when they were recorded.

In [19]:
events = session.get_events(app_name="mini-rag", app_version="v2-top3-loose")
f_new = Metric(judge.conciseness_with_cot_reasons, name="Conciseness (backtest)").on_output()
session.compute_feedbacks_on_events(events, [f_new])
for _ in range(30):                      # results are written as EVAL spans; poll until they land
    session.force_flush()
    recs_bt, _ = session.get_records_and_feedback()
    sel = recs_bt.query("app_version == 'v2-top3-loose'")
    if "Conciseness (backtest)" in sel and sel["Conciseness (backtest)"].notna().all():
        break
    time.sleep(5)
display(sel[["input", "Conciseness (backtest)"]])

,input,Conciseness (backtest)
31,What is the VIN undervoltage lockout hysteresis?,1.000000
32,What RT resistor gives 1 MHz?,0.666667
33,What is the top switch current limit?,1.000000
34,What does the PGOOD pin do?,1.000000
35,What is the thermal resistance theta-JA of the package?,1.000000


## 3.4 Offline / batch evaluation: no app needed

Already have question/answer/context logs, perhaps from another system or a CSV export? `BatchEvaluator` runs metrics over a DataFrame. Selectors become `Selector.from_column("col")`. Results come back **in memory** and are not persisted. For persisted, dashboard-visible evaluation of logs, use a `Run` in `LOG_INGESTION` mode (§3.5).

In [20]:
from trulens.core import BatchEvaluator
logs = pd.DataFrame([
    {"q": "What is VFB?", "a": "VFB is 500 mV typical (495-505 mV).", "ctx": [CORPUS[1]]},
    {"q": "What is VFB?", "a": "VFB is 1.2 V.",                         "ctx": [CORPUS[1]]},     # wrong on purpose
])
batch = BatchEvaluator([
    Metric(judge.groundedness_measure_with_cot_reasons, name="Groundedness").on(
        {"source": Selector.from_column("ctx"), "statement": Selector.from_column("a")}),
    Metric(has_units, name="Has units").on({"answer": Selector.from_column("a")}),
], max_workers=4)
display(batch.evaluate(logs))

Metric implementation <function has_units at 0x00000213ACF828D0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


,q,a,ctx,Groundedness,Groundedness_explanation,Groundedness_latency,Has units,Has units_explanation,Has units_latency
0,What is VFB?,VFB is 500 mV typical (495-505 mV).,"[Electrical characteristics: regulated feedback voltage VFB is 495 mV min, 500 mV typ, 505 mV max over the full temperature range.]",1.0,"{'reasons': [{'criteria': 'VFB is 500 mV typical (495-505 mV).', 'supporting_evidence': 'Criteria: VFB is 500 mV typical (495-505 mV). Supporting Evidence: Electrical characteristics: regulated feedback voltage VFB is 495 mV min, 500 mV typ, 505 mV max over the full temperature range.', 'score':...",4.610650,1.0,{'reason': '2 value(s) with units found'},0.000201
1,What is VFB?,VFB is 1.2 V.,"[Electrical characteristics: regulated feedback voltage VFB is 495 mV min, 500 mV typ, 505 mV max over the full temperature range.]",0.0,"{'reasons': [{'criteria': 'VFB is 1.2 V.', 'supporting_evidence': 'Criteria: VFB is 1.2 V. Supporting Evidence: Electrical characteristics: regulated feedback voltage VFB is 495 mV min, 500 mV typ, 505 mV max over the full temperature range.', 'score': 0.0}]}",6.673784,1.0,{'reason': '1 value(s) with units found'},0.000199


## 3.5 Runs: managed experiments and regression testing

A **Run** is a named execution of an app version over a **dataset**: invocation, then metrics, then status tracking. Two runs over the same inputs can be compared with `run_a.compare(run_b)` (or `compare_runs`), which gives:
- per-item deltas;
- regression flags with a tolerance;
- the aggregate delta with a 95 % CI and a permutation p-value.

This is the statistically sound way to answer "did my change make things worse?". `Mode.APP_INVOCATION` calls your app for each row. `Mode.LOG_INGESTION` ingests existing input/output logs instead.

Run lifecycle: `add_run(RunConfig)` → `start(input_df)` → `compute_metrics([...])` (asynchronous) → `get_status()` / `get_records()` → `compare()`.

> **Local database:** pass `connector=session.connector` to `TruApp`, otherwise `add_run` fails with "Run DAO not initialized". On Snowflake, `start()` also triggers server-side ingestion, so wait for `RunStatus.INVOCATION_COMPLETED` before calling `compute_metrics`.

Below, the grounded prompt (baseline) is compared with the loose prompt (candidate) on the stored golden dataset from §2.6.

In [21]:
from trulens.core.run import RunConfig, compare_runs

def paired_comparison(version_a: str, version_b: str, metric: str, app_name: str = "mini-rag",
                      n_perm: int = 5000, seed: int = 0):
    """Per-question paired delta, bootstrap 95% CI and permutation p-value (what Run.compare reports)."""
    df, _ = session.get_records_and_feedback(app_name=app_name)
    a = df[df.app_version == version_a].set_index("input")[metric]
    b = df[df.app_version == version_b].set_index("input")[metric]
    d = (b - a.reindex(b.index)).dropna().to_numpy()
    rng = np.random.default_rng(seed)
    boots = [rng.choice(d, len(d)).mean() for _ in range(n_perm)]
    perm = [np.mean(d * rng.choice([-1, 1], len(d))) for _ in range(n_perm)]
    p = (np.sum(np.abs(perm) >= abs(d.mean())) + 1) / (n_perm + 1)
    return {"metric": metric, "n": len(d), "mean_delta": round(d.mean(), 3),
            "ci95": tuple(np.round(np.percentile(boots, [2.5, 97.5]), 3)), "p_value": round(p, 4),
            "regressed_items": int((d < 0).sum())}

RUN_METRICS = [f_groundedness, f_correct, f_answer_rel]

def execute_run(app_obj, version: str, run_name: str):
    # connector=session.connector is required for Runs on a local database ("Run DAO not initialized" otherwise)
    tru = TruApp(app_obj, app_name="mini-rag-runs", app_version=version, main_method=app_obj.query,
                 connector=session.connector)
    run = tru.add_run(run_config=RunConfig(run_name=run_name, dataset_name="mini_rag_golden", source_type="DATAFRAME",
                                           dataset_spec={"input": "query", "ground_truth_output": "expected_response"}))
    run.start(input_df=golden_df)                    # invokes the app once per dataset row
    run.compute_metrics(RUN_METRICS)                 # asynchronous; poll the records until all scores are in
    names = [m.name for m in RUN_METRICS]
    for _ in range(90):
        recs = run.get_records()
        if all(n in recs for n in names) and recs[names].notna().all().all():
            break
        time.sleep(5)
    print(run_name, run.get_status(), f"{len(recs)} records")
    return tru, run

tru_base, run_base = execute_run(VERSIONS["v2-top3-grounded"], "v2-top3-grounded", "baseline-grounded")
tru_cand, run_cand = execute_run(VERSIONS["v2-top3-loose"], "v2-top3-loose", "candidate-loose")

diff = run_base.compare(run_cand, tolerance=0.1)      # same as compare_runs(run_base, run_cand)
for name, md_ in diff.metrics.items():
    print(f"{name:<26} Δ={md_.mean_delta:+.2f}  95% CI [{md_.ci_lower:+.2f}, {md_.ci_upper:+.2f}]  "
          f"p={md_.p_value:.3f}  regressed {md_.n_regressed}/{md_.n_items}")
display(pd.DataFrame([{"metric": n, **i.__dict__} for n, m in diff.metrics.items() for i in m.items if i.regressed]))

# the same statistics computed by hand from the leaderboard data (used by the CI gate in §4.7)
print(paired_comparison("v2-top3-grounded", "v2-top3-loose", "Groundedness"))

`app_version` is deprecated. Please use `app_versions` instead.


baseline-grounded RunStatus.INVOCATION_COMPLETED 5 records


`app_version` is deprecated. Please use `app_versions` instead.


`app_version` is deprecated. Please use `app_versions` instead.


`app_version` is deprecated. Please use `app_versions` instead.


candidate-loose RunStatus.INVOCATION_COMPLETED 5 records
Answer relevance           Δ=+0.20  95% CI [+0.00, +0.60]  p=1.000  regressed 0/5
Correctness (vs golden)    Δ=-0.18  95% CI [-0.54, +0.00]  p=1.000  regressed 1/5
Groundedness               Δ=-0.20  95% CI [-0.60, +0.00]  p=1.000  regressed 1/5


,metric,input,score_a,score_b,delta,regressed
0,Correctness (vs golden),What is the thermal resistance theta-JA of the package?,1.0,0.1,-0.9,True
1,Groundedness,What is the thermal resistance theta-JA of the package?,1.0,0.0,-1.0,True


{'metric': 'Groundedness', 'n': 5, 'mean_delta': np.float64(0.0), 'ci95': (np.float64(0.0), np.float64(0.0)), 'p_value': np.float64(1.0), 'regressed_items': 0}


## 3.6 Online evaluation in production: sampling, throttling, budgets

Evaluating 100 % of production traffic with an LLM judge is expensive. `session.configure_online_eval(...)` takes:
- `sample_rate`: a float, or `{app_name: rate}`;
- `throttle`: maximum evaluations per minute;
- `cost_budget`: a daily USD cap, for providers that report cost;
- `feedbacks`: which metrics to run online.

Records carry `sampled`, `sample_rate` and `eval_decision_reason`, so dashboards stay unbiased (re-weight by `1/sample_rate`). Explicit `compute_feedbacks()` calls ignore sampling.

A common production split:
- **online:** cheap deterministic metrics on 100 %, and the groundedness judge on 5–10 %;
- **offline nightly:** the full suite on a curated dataset (§3.8).

In [22]:
session.configure_online_eval(sample_rate={"mini-rag": 0.5}, throttle=60, feedbacks=[f_groundedness])
online_app = VERSIONS["v2-top3-grounded"]
# keep a reference to the TruApp: its background evaluator computes metrics after the `with` blocks end
tru_online = TruApp(online_app, app_name="mini-rag", app_version="v3-online", main_method=online_app.query,
                    feedbacks=[f_groundedness])
for q in QUESTIONS:                       # one `with` per request, as a production request handler would do
    with tru_online:
        online_app.query(q)
session.force_flush(); time.sleep(30)
df_on, _ = session.get_records_and_feedback()
display(df_on.query("app_version == 'v3-online'")[["input", "sampled", "sample_rate", "eval_decision_reason", "Groundedness"]])
session.configure_online_eval(sample_rate=1.0)      # reset to evaluate everything for the rest of the notebook

,input,sampled,sample_rate,eval_decision_reason,Groundedness
51,What is the VIN undervoltage lockout hysteresis?,False,0.5,not_sampled,NaN
52,What RT resistor gives 1 MHz?,False,0.5,not_sampled,NaN
53,What is the top switch current limit?,True,0.5,evaluated,1.0
54,What does the PGOOD pin do?,True,0.5,evaluated,1.0
55,What is the thermal resistance theta-JA of the package?,True,0.5,evaluated,0.0


## 3.7 Guardrails: using metrics *inside* the app

The same metric definitions can gate behaviour at runtime. Each decision is recorded as a GUARDRAIL span.

| Decorator | Effect |
|---|---|
| `@context_filter(metric, threshold, keyword_for_prompt="query")` | Drops retrieved contexts scoring at or below the threshold before generation. Fewer irrelevant chunks means fewer hallucinations. |
| `@block_input(metric, threshold, keyword_for_prompt=…, return_value=…)` | Refuses a request whose input fails the check (e.g. off-topic or malicious) |
| `@block_output(metric, threshold, return_value=…)` | Suppresses an output that fails the check (e.g. harmful) |

Guardrails add latency and cost on the request path. Use fast judges and small thresholds-of-concern.

In [23]:
from trulens.core.guardrails.base import context_filter, block_input

f_ctx_guard = Metric(judge.context_relevance, name="Context relevance (guard)")
f_offtopic = Metric(lambda query: 1.0 if set(tokenize(query)) & {"ltc3310s", "ltc3310", "vin", "vout", "pgood", "rt", "inductor",
                                                                  "switch", "frequency", "uvlo", "vfb"} else 0.0,
                    name="On-topic")

class GuardedRAG(MiniRAG):
    @instrument(span_type=SpanAttributes.SpanType.RETRIEVAL,
                attributes={SpanAttributes.RETRIEVAL.QUERY_TEXT: "query", SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return"})
    @context_filter(feedback=f_ctx_guard, threshold=0.5, keyword_for_prompt="query")
    def retrieve(self, query: str) -> list[str]:
        q = set(tokenize(query))
        return sorted(CORPUS, key=lambda d: -len(q & set(tokenize(d))))[: self.top_k]

    # main_method: TruApp makes it the record root automatically, so no RECORD_ROOT decorator is needed here
    @block_input(feedback=f_offtopic, threshold=0.5, keyword_for_prompt="query",
                 return_value="I can only answer questions about the LTC3310S datasheet.")
    def query(self, query: str) -> str:
        return self.generate(query, self.retrieve(query))

guarded = GuardedRAG(top_k=4)
with TruApp(guarded, app_name="mini-rag", app_version="v4-guarded", main_method=guarded.query,
            feedbacks=[f_answer_rel]) as rec_g:
    print(guarded.query("What is the UVLO hysteresis?"))
    print(guarded.query("Write me a poem about cats"))       # blocked: never reaches retrieval or the LLM
display(rec_g.retrieve_feedback_results(timeout=300))
ev_g = session.get_events(app_name="mini-rag", app_version="v4-guarded")
print("guardrail spans:", int((ev_g["record_attributes"].map(lambda a: a.get(SpanAttributes.SPAN_TYPE)) == "guardrail").sum()))

Metric implementation <function <lambda> at 0x00000213AD6F78A0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


The UVLO hysteresis is **150 mV typical**.
I can only answer questions about the LTC3310S datasheet.


,Answer relevance
record_id,
beb9bb29-c61c-46fa-ae58-48a9aeeddea9,1.0
dec582ff-dd43-4fc4-95b9-4127b8c71d9e,0.0


guardrail spans: 6


## 3.8 From production records to an eval dataset (curation)

Your best test cases are real failures. `session.curate_records_to_dataset(name, records_df, mapping=TraceDatasetMapping(...), expected_response_fn=...)`:
1. turns selected records (for example the lowest-groundedness ones) into ground-truth rows;
2. copies provenance (source record id, app name and version);
3. lets you supply corrected answers.

Your regression dataset then grows with every incident.

In [24]:
from trulens.core.dataset import TraceDatasetMapping
all_recs, _ = session.get_records_and_feedback()
worst = all_recs.dropna(subset=["Groundedness"]).nsmallest(3, "Groundedness")
display(worst[["app_version", "input", "Groundedness"]])

FIXES = {r["query"]: r["expected_response"] for r in golden_df.to_dict("records")}
res = session.curate_records_to_dataset(
    dataset_name="regressions_from_prod", records=worst,
    mapping=TraceDatasetMapping(query="input"),
    expected_response_fn=lambda row: FIXES.get(row["input"], "TODO: write the correct answer"),
    on_error="collect")
print(res)
display(session.get_ground_truth("regressions_from_prod")[["query", "expected_response"]])

,app_version,input,Groundedness
6,v1-top2-grounded,What is the thermal resistance theta-JA of the package?,0.0
25,v2-top1-grounded,What is the thermal resistance theta-JA of the package?,0.0
30,v2-top3-grounded,What is the thermal resistance theta-JA of the package?,0.0


dataset_name='regressions_from_prod' dataset_id='dataset_hash_ee43db1a06a26f615708f672b2109dfa' accepted=3 duplicates=0 rejected=0 ground_truth_ids=['ground_truth_hash_1097328469fb096ec28a77b9cba38d6b', 'ground_truth_hash_042c9f8ca927f07d6d905a52c2b403e6', 'ground_truth_hash_a6ba73b0ccef806667eae39b32301a38'] errors=[]


,query,expected_response
0,What is the thermal resistance theta-JA of the package?,The information is not available.
1,What is the thermal resistance theta-JA of the package?,The information is not available.
2,What is the thermal resistance theta-JA of the package?,The information is not available.


## 3.9 Prompt registry

Version your prompts next to your evals, so every score can be traced back to the exact prompt text:
- `create_prompt(slug)` registers a prompt;
- `create_prompt_version(slug, text, variables, change_note)` adds a version;
- `set_prompt_label(slug, "production", version)` moves a deployment label;
- `get_prompt(slug, label="production")` returns the resolved prompt.

The dashboard's **Prompts** tab shows the history.

In [25]:
p = session.create_prompt("datasheet-answerer", description="System prompt for MiniRAG", tags=["rag", "course"])
v1 = session.create_prompt_version("datasheet-answerer", text="Answer ONLY from the context about {part}.",
                                   variables=["part"], change_note="initial")
v2 = session.create_prompt_version("datasheet-answerer", text="Answer ONLY from the context about {part}. Always give units and MIN/TYP/MAX.",
                                   variables=["part"], change_note="require units + spec type")
session.set_prompt_label("datasheet-answerer", "production", v2)
resolved = session.get_prompt("datasheet-answerer", label="production")
print(resolved)

prompt=Prompt(prompt_id='prompt_hash_89f100f583bad83c488685fac0ff488f', slug='datasheet-answerer', name='datasheet-answerer', prompt_type=<PromptType.TEXT: 'text'>, description='System prompt for MiniRAG', tags=['rag', 'course'], created_at=datetime.datetime(2026, 11, 13, 10, 14, 17, 302896, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 11, 13, 10, 14, 17, 302901, tzinfo=TzInfo(0))) version=PromptVersion(version_id='prompt_version_hash_c447503d2fa7cf22339f055f52d0e7a5', prompt_id='prompt_hash_89f100f583bad83c488685fac0ff488f', prompt_type=<PromptType.TEXT: 'text'>, parent_version_id='prompt_version_hash_872d016d0e16c461c60aae526f242b58', text='Answer ONLY from the context about {part}. Always give units and MIN/TYP/MAX.', messages=None, variables=['part'], model_defaults={}, response_format=None, change_note='require units + spec type', content_hash='content_hash_cbdcbcc3f693a7ff7095f1707c306dd9', created_at=datetime.datetime(2026, 11, 13, 10, 14, 17, 363715, tzinfo=TzInfo(0)),

## 3.10 OpenTelemetry interoperability

TruLens 2.x **is** OpenTelemetry:
- **Export elsewhere:** `TruSession(otel_exporter="otlp", otlp_endpoint="http://localhost:4317")` sends spans to any OTLP backend (Phoenix, Jaeger, Datadog, Grafana Tempo, …). You can also pass `span_exporter=` for a custom exporter.
- **Coexist with another tracer:** OTel allows one global `TracerProvider`. If another tool (e.g. Arize Phoenix) registers first, TruLens adds its span processor to that provider, and both receive spans. This project's eval notebook works that way: agent traces go to Phoenix, and eval records go to TruLens.
- **Raw access:** `session.get_events(app_name, app_version)` returns every span (`record_attributes`, `trace`, timestamps) for your own analysis.
- **Attribute conventions:** `ai.observability.*`, e.g. `ai.observability.retrieval.retrieved_contexts`, `ai.observability.eval_root.score`.

In [26]:
ev = session.get_events(app_name="mini-agent", app_version="agent-v1")
span_types = ev["record_attributes"].map(lambda a: a.get(SpanAttributes.SPAN_TYPE))
print("spans by type:", span_types.value_counts().to_dict())
ex = ev[span_types == "tool"].iloc[0]["record_attributes"]
print({k: (str(v)[:80]) for k, v in ex.items() if k.startswith("ai.observability.call")})

spans by type: {'eval': 14, 'eval_root': 14, 'tool': 9, 'generation': 8, 'record_root': 2}
{'ai.observability.call.return': 'Oscillator: default switching frequency with RT tied to VIN is 2 MHz typ (1.8 MH', 'ai.observability.call.function': '__main__.MiniAgent.lookup_spec', 'ai.observability.call.kwargs.query': 'RT resistor for 1 MHz switching frequency'}


## 3.11 Cost and latency

The leaderboard and records include `latency`, `total_tokens` and `total_cost` for app calls TruLens can see: OpenAI, LiteLLM, Bedrock and Cortex calls made inside the record. Evaluation cost is reported separately (`eval_cost`, `<metric> feedback cost in USD`).

Calls the instrumentation can't see report 0: a custom HTTP client, or work on another thread. Record your own token counts as span attributes in that case.

Make cost a first-class metric: track **quality per dollar** and **p95 latency** next to quality scores. A version that gains +0.02 groundedness for 3× cost is usually a regression.

In [27]:
recs_all, _ = session.get_records_and_feedback()
summary = (recs_all.groupby(["app_name", "app_version"])
           .agg(records=("record_id", "count"), p50_latency=("latency", "median"),
                p95_latency=("latency", lambda s: s.quantile(0.95)), tokens=("total_tokens", "mean"),
                app_cost=("total_cost", "sum"), eval_cost=("eval_cost", "sum")))
display(summary.round(4))

records  p50_latency  p95_latency  tokens  \
app_name      app_version                                                    
mini-agent    agent-v1                 2       9.8244      10.5828     0.0   
mini-chat     chat-v1                  3       1.8785       1.9115     0.0   
mini-rag      v0-no-metrics            2       1.8149       1.8962     0.0   
              v1-top2-grounded         5       1.7398       1.8689     0.0   
              v1b-rerank               2       1.8901       2.0386     0.0   
              v1c-selectors            2       1.8609       1.9913     0.0   
              v1d-quality              2       1.4446       1.5013     0.0   
              v1e-custom-judges        3       1.7838       2.4356     0.0   
              v1f-deterministic        5       1.7200       2.1984     0.0   
              v2-top1-grounded         5       1.9379       2.2617     0.0   
              v2-top3-grounded         5       1.9021       2.7367     0.0   
              v2-top3-loose            5       2.1832       3.8198     0.0   
              v3-online                5       2.4125       2.7319     0.0   
              v4-guarded               2       2.0527       3.8995     0.0   
mini-rag-runs v2-top3-grounded         5       1.8668       2.6964     0.0   
              v2-top3-loose            5       2.5108       2.8430     0.0   

                                 app_cost  eval_cost  
app_name      app_version                             
mini-agent    agent-v1                0.0        0.0  
mini-chat     chat-v1                 0.0        0.0  
mini-rag      v0-no-metrics           0.0        0.0  
              v1-top2-grounded        0.0        0.0  
              v1b-rerank              0.0        0.0  
              v1c-selectors           0.0        0.0  
              v1d-quality             0.0        0.0  
              v1e-custom-judges       0.0        0.0  
              v1f-deterministic       0.0        0.0  
              v2-top1-grounded        0.0        0.0  
              v2-top3-grounded        0.0        0.0  
              v2-top3-loose           0.0        0.0  
              v3-online               0.0        0.0  
              v4-guarded              0.0        0.0  
mini-rag-runs v2-top3-grounded        0.0        0.0  
              v2-top3-loose           0.0        0.0

# Part 4: Expert, evaluating the evaluator

## 4.1 Why meta-evaluation

An LLM judge is a model with its own error rate. Before you gate releases on a metric, measure:
- **agreement with humans**: Cohen's κ, Pearson or Spearman correlation on a labelled sample;
- **consistency**: the same input judged several times; look at the variance;
- **bias**: position, verbosity, self-preference, sensitivity to style rather than substance.

TruLens provides `GroundTruthAggregator` (agreement statistics), `Jury` (ensembles) and `FewShotOptimizer` (calibration). Below, a small labelled set of answers to *"What is the UVLO hysteresis?"*, with human groundedness labels in {0, 1}.

In [28]:
from trulens.feedback import GroundTruthAggregator

SOURCE = CORPUS[2]
LABELLED = [  # (statement, human label: 1 = fully supported by SOURCE)
    ("The UVLO hysteresis is 150 mV typical.", 1),
    ("UVLO rising threshold is 2.1 V typical with 150 mV of hysteresis.", 1),
    ("The UVLO hysteresis is 150 mV, guaranteed over temperature.", 0),     # 'guaranteed' is not supported
    ("The UVLO hysteresis is 200 mV.", 0),
    ("UVLO trips at 2.2 V max on the rising edge.", 1),
    ("The falling UVLO threshold is exactly 1.95 V minimum.", 0),          # derived and mislabelled as min
    ("Hysteresis on UVLO is about 0.15 V.", 1),
    ("The UVLO hysteresis depends on the RT resistor.", 0),
]
human = [y for _, y in LABELLED]
judge_scores = [judge.groundedness_measure_with_cot_reasons(SOURCE, s)[0] for s, _ in LABELLED]

agg = GroundTruthAggregator(true_labels=human)
print("judge scores :", [round(x, 2) for x in judge_scores])
print("human labels :", human)
print("Cohen's kappa:", round(agg.cohens_kappa([1 if s >= 0.5 else 0 for s in judge_scores]), 3),
      "| Pearson:", round(agg.pearson_correlation(judge_scores), 3),
      "| AUC:", round(agg.auc(judge_scores), 3),
      "| Brier:", round(agg.brier_score(judge_scores), 3))

judge scores : [1.0, 1.0, 0.33, 0.0, 1.0, 0.0, 1.0, 0.0]
human labels : [1, 1, 0, 0, 1, 0, 1, 0]
Cohen's kappa: 1.0 | Pearson: 0.976 | AUC: 1.0 | Brier: 0.014


## 4.2 Consistency: run the same judgement several times

Reasoning models aren't perfectly deterministic. Measure the spread per item. High variance means you need more samples (a `Jury` with repetition), a stronger judge, or a clearer rubric.

In [29]:
REPEATS = 3
spread = []
for s, y in LABELLED[:4]:
    xs = [judge.groundedness_measure_with_cot_reasons(SOURCE, s)[0] for _ in range(REPEATS)]
    spread.append({"statement": s[:60], "human": y, "scores": [round(x, 2) for x in xs], "std": round(float(np.std(xs)), 3)})
display(pd.DataFrame(spread))

,statement,human,scores,std
0,The UVLO hysteresis is 150 mV typical.,1,"[1.0, 1.0, 1.0]",0.000
1,UVLO rising threshold is 2.1 V typical with 150 mV of hyster,1,"[1.0, 1.0, 1.0]",0.000
2,"The UVLO hysteresis is 150 mV, guaranteed over temperature.",0,"[0.33, 0.67, 0.33]",0.157
3,The UVLO hysteresis is 200 mV.,0,"[0.0, 0.0, 0.0]",0.000


## 4.3 Juries: ensembles of judges

`Jury(jurors=[providers…], method="<provider method name>", aggregation="mean" | "median" | "majority" | callable, weights=…, threshold=…)` calls the same metric on several judges in parallel and aggregates the scores.

Its result includes reliability statistics (`reliability.scores`, `reliability.n_scores`, …), which reach the EVAL span and the dashboard. Juries reduce single-model bias and variance. Mix model families when you can, e.g. GPT plus Claude through LiteLLM. `Jury.repeated(provider, n, method)` builds a self-consistency jury from one judge.

In [30]:
from trulens.feedback import Jury
jury = Jury(jurors=[ResponsesJudge("gpt-5.4-mini", "low"), ResponsesJudge("gpt-5.4-mini", "medium"),
                    ResponsesJudge("gpt-5.5", "low")],
            method="groundedness_measure_with_cot_reasons", aggregation="median")
score, meta = jury(source=SOURCE, statement=LABELLED[2][0])
print("jury score:", score); print({k: v for k, v in meta.items() if k.startswith("reliability")})

f_ground_jury = Metric(jury, name="Groundedness (jury)").on(
    {"source": Selector.select_context(collect_list=True), "statement": Selector.select_record_output()})

jury score: 0.3333333333333333
{'reliability.n_scores': 3, 'reliability.scores': [0.3333333333333333, 0.3333333333333333, 0.3333333333333333], 'reliability.score_std': 0.0, 'reliability.flip_rate': 0.0, 'reliability.outcome_entropy': 0.0, 'reliability.temperature': None}


## 4.4 Calibrating a judge with few-shot examples

`FewShotOptimizer(feedback_fn, candidates, eval_dataset, n_examples, metric="pearson" | "cohens_kappa" | "f1" | …)` greedily picks the labelled examples that, injected into the judge prompt, **maximize agreement with human labels**. Pass the result through `examples=` on the metric.

The data format is `LabeledExample = (kwargs_for_feedback_fn, score)`. Keep candidates and eval data disjoint, as you would avoid train/test leakage.

In [31]:
from trulens.feedback import FewShotOptimizer

def grounded_fn(source: str, statement: str, examples: str | None = None) -> float:
    """A small custom groundedness judge whose prompt accepts few-shot examples (injected by the optimizer)."""
    system = ("Rate 0-3 how fully the STATEMENT is supported by the SOURCE. 3 = every detail supported, including whether "
              "a value is typical or guaranteed; 0 = contradicted or unsupported.")
    if examples:
        system += f"\n\nCalibration examples:\n{examples}"
    return judge.generate_score(system_prompt=system, user_prompt=f"SOURCE: {source}\nSTATEMENT: {statement}",
                                min_score_val=0, max_score_val=3)

candidates = [({"source": SOURCE, "statement": s}, y) for s, y in LABELLED[:4]]
eval_set   = [({"source": SOURCE, "statement": s}, y) for s, y in LABELLED[4:]]
try:
    opt = FewShotOptimizer(feedback_fn=grounded_fn, candidates=candidates, eval_dataset=eval_set,
                           n_examples=2, metric="pearson", max_workers=4)
    result = opt.optimize()
    print("best examples:", [e[0]["statement"] for e in result.best_examples])
    print("agreement:", result)
except Exception as e:
    print("optimizer:", type(e).__name__, str(e)[:300])

best examples: ['The UVLO hysteresis is 150 mV typical.', 'UVLO rising threshold is 2.1 V typical with 150 mV of hysteresis.']
agreement: OptimizeResult(best_examples=[({'source': 'Electrical characteristics: VIN undervoltage lockout (VIN rising) 2.0 V min, 2.1 V typ, 2.2 V max; UVLO hysteresis 150 mV typical.', 'statement': 'The UVLO hysteresis is 150 mV typical.'}, 1), ({'source': 'Electrical characteristics: VIN undervoltage lockout (VIN rising) 2.0 V min, 2.1 V typ, 2.2 V max; UVLO hysteresis 150 mV typical.', 'statement': 'UVLO rising threshold is 2.1 V typical with 150 mV of hysteresis.'}, 1)], correlation=0.8164965809277261, candidate_scores={0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0}, metric_name='pearson', metric_score=0.8164965809277261)


## 4.5 Bias probes you should run on any judge

| Bias | Probe | Mitigation |
|---|---|---|
| **Verbosity** | Same content, padded with filler → score should not rise | Rubric that rewards concision; `conciseness` as a separate metric |
| **Position** (pairwise judging) | Swap A/B order → verdict should flip with it | Judge both orders; average |
| **Self-preference** | Judge = generator model family | Use a different family, or a jury |
| **Authority / style** | Confident tone or a fake citation → score should not rise | `citation_accuracy`; groundedness against the real source |
| **Leniency drift** | Scores creep up across model upgrades | Freeze the judge model and version; re-calibrate on the labelled set (§4.1) after changes |

In [32]:
base = "The UVLO hysteresis is 150 mV typical."
padded = base + " " + "This is an important parameter that engineers should carefully consider in their designs. " * 4
confident = "According to the official datasheet, page 3, the UVLO hysteresis is definitely 200 mV."
for name, stmt in [("base", base), ("padded", padded), ("confident-wrong", confident)]:
    print(f"{name:16s} groundedness={judge.groundedness_measure_with_cot_reasons(SOURCE, stmt)[0]:.2f}  "
          f"relevance={judge.relevance('What is the UVLO hysteresis?', stmt):.2f}")

base             groundedness=1.00  relevance=1.00


padded           groundedness=1.00  relevance=1.00


confident-wrong  groundedness=0.00  relevance=1.00


## 4.6 Designing an evaluation suite

**Choose metrics from failure modes, not from the catalog:**

| Failure mode | Metric(s) |
|---|---|
| Retrieval misses the fact | context relevance, retrieval recall vs expected chunks |
| Hallucination | groundedness (+ answerability variant) |
| Wrong answer although grounded | correctness vs golden |
| Off-target answer | answer relevance |
| Agent loops / wrong tools | tool selection, tool calling, execution efficiency, plan adherence |
| Formatting / contract breaks | deterministic: schema, regex, citations |
| Unsafe output | harmfulness family (`higher_is_better=False`) |
| Too slow / too expensive | latency p95, tokens, cost per query |

**Dataset design:**
- Stratify by category: lookups, multi-hop, calculations, comparisons, unanswerable, adversarial.
- Include **negative** cases, where the right answer is "I don't know".
- Use 30–100 items for a stable signal. With 10, a 0.1 change is often noise, so check the p-value (§3.5).

**Process:**
1. Write the golden set.
2. Label 30+ judge decisions by hand.
3. Measure judge agreement (κ ≥ 0.6) before trusting the metric.
4. Freeze the judge model, effort and prompts.
5. Version everything: app, prompts, dataset, judge.

## 4.7 CI gating

Fail the build when quality regresses. Pattern: run the suite on the golden set, then assert **absolute floors** and **no significant regression** against the last released version.

In [33]:
GATES = {"Groundedness": 0.80, "Correctness (vs golden)": 0.70, "Answer relevance": 0.80}

def ci_gate(candidate: str, baseline: str | None = None, alpha: float = 0.05, app_name: str = "mini-rag") -> bool:
    df, _ = session.get_records_and_feedback(app_name=app_name)
    cand = df[df.app_version == candidate]
    ok = True
    for metric, floor in GATES.items():
        mean = cand[metric].mean()
        status = "PASS" if mean >= floor else "FAIL"
        ok &= mean >= floor
        line = f"{status} {metric:<26} mean={mean:.2f} floor={floor}"
        if baseline:
            cmp = paired_comparison(baseline, candidate, metric)
            if cmp["mean_delta"] < 0 and cmp["p_value"] < alpha:
                ok = False; line += f"  REGRESSION Δ={cmp['mean_delta']} p={cmp['p_value']}"
        print(line)
    return ok

print("v2-top3-grounded vs v2-top1-grounded ->", ci_gate("v2-top3-grounded", baseline="v2-top1-grounded"))
print("v2-top3-loose    vs v2-top3-grounded ->", ci_gate("v2-top3-loose", baseline="v2-top3-grounded"))
# In CI:  assert ci_gate(candidate, baseline), "quality gate failed"

PASS Groundedness               mean=0.80 floor=0.8


PASS Correctness (vs golden)    mean=0.98 floor=0.7


PASS Answer relevance           mean=0.80 floor=0.8
v2-top3-grounded vs v2-top1-grounded -> True


PASS Groundedness               mean=0.80 floor=0.8


PASS Correctness (vs golden)    mean=0.82 floor=0.7


PASS Answer relevance           mean=1.00 floor=0.8
v2-top3-loose    vs v2-top3-grounded -> True


## 4.8 Production checklist

- **Storage:** Postgres (or Snowflake) instead of SQLite for shared, concurrent use; run `migrate_database()` on upgrades.
- **Judge hygiene:** pin model *and* version; record the judge name in `app_version` or metadata; re-calibrate after any judge change.
- **Throughput:** `Endpoint(rpm=…)` rate-limits judge calls; `BatchEvaluator(max_workers=…)`; `RunConfig(metric_max_workers=…)`.
- **Cost control:** online sampling + throttle + `cost_budget`; deterministic metrics first; cheaper judge for high-volume metrics, stronger judge for the few that gate releases.
- **Privacy:** traces contain prompts and outputs. Treat the TruLens DB like production logs (access control, retention). Bind the dashboard to localhost or put it behind auth.
- **Observability:** export spans over OTLP to your tracing backend, so eval scores and traces live side by side.
- **Feedback loop:** curate failing production records into datasets (§3.8), so every incident becomes a regression test.

# Part 5: Exercises

1. **Selectors:** write a metric that reads the `mini_rag.rerank.dropped` attribute and penalizes dropping more than 2 contexts.
2. **Custom judge:** build a "units correctness" judge with `generate_score_and_reasons` that checks unit conversions (mV vs V).
3. **Groundedness:** compare `Groundedness` and `Groundedness (answerability)` on the θJA question for v2-top3-grounded vs v2-top3-loose, and explain the difference.
4. **Agents:** add a deliberately bad agent version (remove the calculator tool) and see which agent metric catches it first.
5. **Meta-eval:** label 20 more groundedness cases, compute κ for `gpt-5.4-mini` vs a `Jury`, and decide which to use in CI.
6. **Regression:** make a prompt change, run it as a new version, and use `paired_comparison` / `ci_gate` to decide whether to ship.
7. **Real system:** apply what you learned to `trulens_evals.ipynb`: add agent trace metrics to the datasheet agent and a judge-calibration section.